# Part 1 (balanced): nouns / adjectives / verbs, one LoRA per language (r=4)
Lemma-disjoint 80/10/10 split, then **equal rows for N, ADJ and V** in train, val and test. Run the setup cell, then the main cell. The last cell (extra) splits the test results by word-frequency tier.

In [ ]:
%pip install -q --upgrade torchao
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os, json, tempfile
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

# ---------------- settings ----------------
LANGS = ['eng', 'spa', 'swe', 'fin']
FILES = {'fin': ['fin.1', 'fin.2']}       # Finnish is split in two UniMorph files
N_TRAIN, N_EVAL = 5000, 1000              # rows PER POS (N, ADJ, V): train / val and test
RANK, EPOCHS, SEED = 4, 6, 42
MODEL, MAXLEN, BS, LR = 'google/byt5-small', 64, 32, 3e-3
OUT = '/content/drive/MyDrive/lora/definitive' if IN_COLAB else '../data/definitive'
os.makedirs(f'{OUT}/models', exist_ok=True)

# ---------------- data: lemma split first, then EQUAL ROWS per POS in every split ----------------
def balance(d, n):
    n = min(n, d.pos.value_counts().min())          # same number of rows for N, ADJ and V
    return pd.concat(g.sample(n, random_state=SEED) for _, g in d.groupby('pos')).reset_index(drop=True)

def prepare(lang):
    url = 'https://raw.githubusercontent.com/unimorph/{}/master/{}'
    df = pd.concat([pd.read_csv(url.format(lang, f), sep='\t', header=None, names=['lemma', 'target', 'features'],
                                quoting=3, keep_default_na=False, dtype=str, on_bad_lines='skip')
                    for f in FILES.get(lang, [lang])]).apply(lambda c: c.str.strip())
    df = df[(df != '').all(axis=1) & ~df.target.str.contains('—', regex=False)]
    if lang == 'eng':                                # Wiktionary markup instead of forms
        df = df[~(df.target.isin(['countable', 'uncountable', 'comparable', 'comparative', 'superlative']) & (df.target != df.lemma))]
    df['pos'] = df.features.str.split(';').str[0].str.split('.').str[0]    # V.PTCP -> V
    df = df[df.pos.isin(['N', 'ADJ', 'V'])].drop_duplicates()
    # one row per (lemma, features); all attested forms kept in 'valid' for multi-valid scoring
    df = df.groupby(['lemma', 'features', 'pos'], as_index=False).target.agg(list).rename(columns={'target': 'valid'})
    df['target'] = df.valid.str[0]
    # 80/10/10 split by lemma (no lemma in two splits, also across POS)
    lem = df.lemma.drop_duplicates().sample(frac=1, random_state=SEED).tolist()
    a, b = int(.8 * len(lem)), int(.9 * len(lem))
    df['split'] = df.lemma.map({l: s for s, ls in zip(['train', 'val', 'test'], [lem[:a], lem[a:b], lem[b:]]) for l in ls})
    out = {s: balance(df[df.split == s], n) for s, n in [('train', N_TRAIN), ('val', N_EVAL), ('test', N_EVAL)]}
    for s in out:
        print(lang, s, 'rows', out[s].pos.value_counts().to_dict(), '| lemmas', out[s].groupby('pos').lemma.nunique().to_dict())
    return out

data = {l: prepare(l) for l in LANGS}
for l, d in data.items():                           # lemma-disjoint train / val / test
    tr, va, te = (set(d[s].lemma) for s in ['train', 'val', 'test'])
    assert not (tr & va or tr & te or va & te), f'lemma leakage in {l}'
print('OK: train / val / test share no lemma, in any language.')

# ---------------- training / evaluation ----------------
tok = AutoTokenizer.from_pretrained(MODEL)
src = lambda d: (d.features + ' ' + d.lemma).tolist()      # Ale's input_v2_features_first

def train_eval(train, val, test, rank, save=None):
    def enc(d):
        x = dict(tok(src(d), max_length=MAXLEN, truncation=True))
        x['labels'] = tok(text_target=d.target.tolist(), max_length=MAXLEN, truncation=True)['input_ids']
        return Dataset.from_dict(x)

    def metric(ep):                                  # multi-valid exact match on val, used to pick the best epoch
        p = ep.predictions[0] if isinstance(ep.predictions, tuple) else ep.predictions
        p = tok.batch_decode(np.where(p >= 0, p, tok.pad_token_id), skip_special_tokens=True)
        return {'exact_match': float(np.mean([a.strip() in v for a, v in zip(p, val.valid)]))}

    model = get_peft_model(AutoModelForSeq2SeqLM.from_pretrained(MODEL), LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=rank, lora_alpha=2 * rank, lora_dropout=0.1, bias='none',
        target_modules=['q', 'k', 'v', 'o', 'wi_0', 'wi_1', 'wo']))
    args = Seq2SeqTrainingArguments(
        output_dir=tempfile.mkdtemp(), eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='exact_match', greater_is_better=True,
        learning_rate=LR, per_device_train_batch_size=BS, per_device_eval_batch_size=BS * 2, num_train_epochs=EPOCHS,
        seed=SEED, predict_with_generate=True, generation_max_length=MAXLEN,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(), report_to='none', logging_steps=100)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=enc(train), eval_dataset=enc(val),
                             data_collator=DataCollatorForSeq2Seq(tok, model=model, pad_to_multiple_of=8),
                             compute_metrics=metric, processing_class=tok)
    trainer.train()
    hist = [(h['epoch'], h['eval_exact_match']) for h in trainer.state.log_history if 'eval_exact_match' in h]

    model.eval(); texts, preds = src(test), []
    for i in range(0, len(texts), 64):
        b = tok(texts[i:i + 64], return_tensors='pt', padding=True, max_length=MAXLEN, truncation=True).to(model.device)
        with torch.no_grad():
            preds += tok.batch_decode(model.generate(**b, max_length=MAXLEN), skip_special_tokens=True)
    if save: model.save_pretrained(save)
    out = test.assign(pred=[p.strip() for p in preds])
    out['ok'] = [p in v for p, v in zip(out.pred, out.valid)]      # multi-valid exact match
    out['strict'] = out.pred == out.target                          # exact match with the first listed form
    del model, trainer; torch.cuda.empty_cache()
    return out, hist

# ---------------- one LoRA per language on the balanced data ----------------
res, hist = [], {}
for l in LANGS:
    f, h = f'{OUT}/p1_{l}_test.parquet', f'{OUT}/p1_{l}_hist.json'
    if os.path.exists(f):                            # finished in an earlier run: reuse, do not retrain
        r = pd.read_parquet(f)
        if os.path.exists(h): hist[l] = json.load(open(h))
        print(f'{l}: reusing saved results from {f}')
    else:
        d = data[l]
        r, hist[l] = train_eval(d['train'], d['val'], d['test'], RANK, save=f'{OUT}/models/{l}_r{RANK}')
        r = r.assign(lang=l); json.dump(hist[l], open(h, 'w')); r.to_parquet(f)   # parquet last = language is complete
    res.append(r)
res = pd.concat(res)

acc = {c: res.pivot_table(index='lang', columns='pos', values=c)[['N', 'ADJ', 'V']].loc[LANGS] for c in ['ok', 'strict']}
print(f'\nTest accuracy by POS (multi-valid, r={RANK}, {N_EVAL} test rows per POS):'); print(acc['ok'].round(3))
print('\nStrict:'); print(acc['strict'].round(3))
print('\nSome wrong predictions:')
print(res[~res.ok].groupby(['lang', 'pos']).head(3)[['lang', 'pos', 'features', 'lemma', 'pred', 'target']].to_string(index=False))

# ---------------- plots ----------------
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
pd.DataFrame({l: data[l]['train'].groupby('pos').lemma.nunique() for l in LANGS}).loc[['N', 'ADJ', 'V']].T.plot.bar(
    ax=ax[0], rot=0, title=f'Train: distinct lemmas per POS\n(rows are equal: {N_TRAIN} each)')
acc['ok'].plot.bar(ax=ax[1], rot=0, ylim=(0, 1.05), title='Test accuracy by POS (multi-valid)')
for l in hist: ax[2].plot(*zip(*hist[l]), marker='o', label=l)
ax[2].set(title='Validation accuracy per epoch', xlabel='epoch', ylim=(0, 1.05), xticks=range(1, EPOCHS + 1)); ax[2].legend()
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/p1_summary.png', dpi=150); plt.show()

## Extra: accuracy by frequency tier

In [ ]:
%pip install -q wordfreq
from wordfreq import zipf_frequency

# Tier of the lemma by word frequency (Zipf scale, same cut-offs as before): common >= 3.5 > mid >= 1.0 > obscure
CODE = {'eng': 'en', 'spa': 'es', 'swe': 'sv', 'fin': 'fi'}
res['zipf'] = [zipf_frequency(w, CODE[l]) for w, l in zip(res.lemma, res.lang)]
res['tier'] = pd.cut(res.zipf, [-1, 1.0, 3.5, 100], labels=['obscure', 'mid', 'common'], right=False).astype(str)
T = ['common', 'mid', 'obscure']

g = res.groupby(['lang', 'pos', 'tier'])
acc = g.ok.mean().unstack('tier').reindex(columns=T).round(3)
print('Test accuracy (multi-valid) by language x POS x frequency tier:'); print(acc)
print('\nTest rows behind each cell:'); print(g.size().unstack('tier').reindex(columns=T).fillna(0).astype(int))
print('\nBy tier only:'); print(res.groupby(['lang', 'tier']).ok.mean().unstack('tier').reindex(columns=T).round(3))

fig, ax = plt.subplots(1, len(LANGS), figsize=(4.5 * len(LANGS), 4), sharey=True)
for a, l in zip(ax, LANGS):
    acc.loc[l].reindex(['N', 'ADJ', 'V']).plot.bar(ax=a, rot=0, ylim=(0, 1.05), title=l, width=.8)
    a.grid(alpha=.3)
ax[0].set_ylabel('test accuracy (multi-valid)')
plt.tight_layout(); plt.savefig(f'{OUT}/p1_tiers.png', dpi=150); plt.show()